# 🦷 Fine-grained Classification of Surgical Instruments — DINOv2 + Length Fusion + ArcFace

Pipeline for classifying **14 surgical instrument classes** (~400 images) on a green cloth with shadows. Some classes differ only by length:

```
image ──► DINOv2 ViT-S/14 ──► CLS embedding (384-dim)──┐
                                                      ├─ concat(385) ─► Linear ─► 384 ─► ArcFace loss
mask ──► minAreaRect ──► length (normalized) ─────────┘
```

**What this notebook does:** install deps → write all 6 modules (`config/dataset/model/train/evaluate/infer`) → sanity-check data → train (LoRA + warmup/cosine + early stopping) → confusion matrix → inference demo

**Data to prepare:** Roboflow export as *COCO Segmentation* with structure
`DATA_DIR/train/_annotations.coco.json` + `DATA_DIR/valid/_annotations.coco.json`


## 0) Install dependencies

In [ ]:
%pip install -q -U torchao peft transformers pytorch-metric-learning albumentations opencv-python-headless scikit-learn seaborn tqdm
print('✅ deps ready')

## 1) Prepare data — pick **one method** and run this cell

In [ ]:
# ── Set DATA_DIR to where your dataset lives ──────────────────────────────
DATA_DIR = "/content/dental_dataset"

# ▸ Method A: Upload a .zip file (exported from Roboflow and zipped)
# from google.colab import files
# up = files.upload()                       # select dataset.zip
# !unzip -o -q "{list(up.keys())[0]}" -d /content/
# DATA_DIR = "/content/dataset"             # ← adjust to the folder name inside the zip

# ▸ Method B: Dataset already on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/path/to/dataset"

# ▸ Method C: Pull directly from Roboflow (set API_KEY / WORKSPACE / PROJECT / VERSION)
# %pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# rf.workspace("WORKSPACE").project("PROJECT").version(VERSION)\
#   .download("coco-segmentation", location=DATA_DIR)

import pathlib
assert pathlib.Path(DATA_DIR, "train", "_annotations.coco.json").exists(), \
    f"not found {DATA_DIR}/train/_annotations.coco.json — run the data-import cell first"
for sp in ["train", "valid", "test"]:
    p = pathlib.Path(DATA_DIR, sp)
    if p.exists():
        n_img = len(list(p.glob('*.jpg'))) + len(list(p.glob('*.png'))) + len(list(p.glob('*.jpeg')))
        n_json = len(list(p.glob('*annotations*.json')))
        print(f"{sp:6s}: {n_img:4d} images, {n_json} annotation file(s)")

## 2) Write module files (%%writefile)

Each cell creates a .py file by role — you can edit the code from the file editor on the left side of Colab after running

In [ ]:
%%writefile config.py
# -*- coding: utf-8 -*-
"""
config.py — central defaults for the whole pipeline

Edit values here, or override when creating the object: TrainConfig(data_dir=..., epochs=...)
"""
from dataclasses import asdict, dataclass
from typing import List, Optional


@dataclass
class TrainConfig:
    # ---------------- Data ----------------
    data_dir: str = "dataset"          # folder with train/ and valid/ (Roboflow COCO Segmentation export)
    img_size: int = 504                # must be divisible by 14 (504=36×14) — from kNN probe experiments
    val_fraction: float = 0.2          # used when valid/ folder is missing → stratified split from train
    calibration_ratio: Optional[float] = None  # cm/pixel — measured from a reference object of known size
                                               # (camera rig is fixed, so one ratio works for all images)
                                               # None = use pixel units, normalized by train mean/std
    flip_allowed: Optional[List[str]] = None   # class names that are allowed to be flipped
                                               # None = all classes can be flipped
                                               # remove handedness classes (left/right) from the list
    num_workers: int = 2               # Colab/Linux can use 2 — on Windows set to 0 if it hangs
    bbox_margin: float = 0.15          # crop margin around bbox per instance (0=no crop, from experiments)

    # ---------------- Model ----------------
    backbone_name: str = "facebook/dinov2-small"  # ViT-S/14, hidden dim = 384 (~21M params)
    finetune_mode: str = "lora"        # "lora" (recommended for small data) | "partial" | "frozen"
    partial_last_blocks: int = 2       # for mode="partial": unfreeze last N ViT blocks + final LayerNorm
    lora_r: int = 16                   # from experiments (r16 better than r8 on this data)
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    head_dropout: float = 0.1          # dropout for fusion head
    use_attention_pool: bool = True    # True = AttentionPooling instead of CLS-only (v2)
                                      # False = use CLS token (backward compat)
    mixup_alpha: float = 0.4           # Mixup alpha for regularization (0.0 = off)
                                      # smaller = stronger, 0.4 suits ~30 samples/class
    use_tta: bool = True               # enable TTA at inference (flip + multi-scale)
    # ---------------- SEF (Scharr Edge Fusion) ----------------
    use_sef: bool = False                # enable Scharr edge branch (adds 64-dim edge feature)

    # ---------------- ArcFace ----------------
    margin: float = 28.6               # additive angular margin in degrees (≈ 0.5 rad)
                                       # — pytorch-metric-learning expects degrees and converts to radians
    scale: float = 64.0                # s: scale cosine before softmax so gradients don't vanish

    # ---------------- CAHM (Confusion-Aware Hard Mining) ----------------
    use_cahm: bool = False
    cahm_alpha: float = 2.0            # extra weight for confused pairs
    cahm_beta: float = 0.9             # EMA smoothing for difficulty score
    cahm_start_epoch: int = 10         # start after this epoch (let confusion stabilize)

    # ---------------- LGMS (Length-Gated Margin Scaling) ----------------
    use_lgms: bool = False
    lgms_gamma: float = 10.0           # max extra degrees for margin of length-similar classes
    lgms_k: int = 2                    # number of nearest twin classes

    # ---------------- Training ----------------
    batch_size: int = 32               # fills T4 15GB at 504px with DINOv2-S + LoRA
    epochs: int = 50
    patience: int = 12                 # early stopping: stop when val accuracy doesn't improve for N epochs
    lr_head: float = 3e-4              # learning rate for fusion head
    lr_backbone: float = 5e-6          # for finetune_mode="partial"
    lr_lora: float = 1e-4              # for finetune_mode="lora"
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1          # first 10% of total steps is linear warmup, then cosine decay
    grad_clip: float = 1.0
    seed: int = 42
    kfold: Optional[int] = None        # e.g. 5 = Stratified 5-fold CV (more reliable for small data)
    output_dir: str = "outputs"

    def to_dict(self) -> dict:
        """Convert config to dict (saved to checkpoint for reproducibility at evaluate/infer)"""
        return asdict(self)


In [ ]:
%%writefile dataset.py
# -*- coding: utf-8 -*-
"""
dataset.py — Load images + segmentation masks from COCO format (Roboflow export)

Main responsibilities:
1) parse ``_annotations.coco.json`` → records (path, polygon, label)
2) rasterize polygon → binary mask
3) measure instrument length from mask (minAreaRect) as a single auxiliary feature
4) task-safe augmentation:
   - focus on photometric ops (brightness/contrast/gamma/CLAHE) to simulate
     specular reflections on metal and varying illumination on the green cloth
   - no crop/zoom that would destroy aspect ratio or absolute scale
     (size is a key discriminative feature!)
   - no cutout / random erasing over the instrument
   - horizontal flip can be toggled per class (some classes have handedness
     and must not be flipped)

Notes:
- Background is green surgical cloth; shadows cast on the cloth move with
  instrument placement / light direction and make bounding/segmentation harder
  (harder than a high-contrast silver tray). Photometric + shadow simulation
  targets this difficulty.
- Experimental defaults found useful elsewhere in the project: image size 504
  and bbox margin ~0.15. Defaults in this module are kept for compatibility;
  see Dataset docstring.
"""
import json
import math
import os
import random
from typing import List, Optional, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN: Tuple[float, ...] = (0.485, 0.456, 0.406)
IMAGENET_STD: Tuple[float, ...] = (0.229, 0.224, 0.225)


# ============================================================ Length measurement from mask
def measure_length_px(mask: np.ndarray) -> float:
    """
    Return the maximum length of the instrument in pixels (from a single binary mask).

    Uses ``cv2.minAreaRect`` because instruments are often placed diagonally
    rather than axis-aligned — the minimum-area rotated rectangle enclosing the
    contour gives a long side that approximates the true instrument length.
    """
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:  # empty mask (annotation error) → return 0 to avoid crash
        return 0.0
    largest = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(largest)  # ((cx,cy), (w,h), angle)
    (w, h) = rect[1]
    return float(max(w, h))


def get_length_cm(mask: np.ndarray, calibration_ratio: float) -> float:
    """Convert pixel length → cm using calibration_ratio (cm/pixel) from a reference object."""
    return measure_length_px(mask) * calibration_ratio


def mask_from_coco_segmentation(segmentation, height: int, width: int) -> np.ndarray:
    """
    Convert COCO segmentation → binary mask (uint8, values 0/255).

    Supports polygon (standard Roboflow format) and RLE (requires pycocotools).
    """
    mask = np.zeros((height, width), dtype=np.uint8)
    if isinstance(segmentation, dict):  # RLE format
        try:
            from pycocotools import mask as mask_utils
        except ImportError as exc:
            raise ImportError(
                "Found RLE segmentation but pycocotools is not installed (pip install pycocotools)"
            ) from exc
        rle = segmentation
        if isinstance(rle.get("counts"), list):  # uncompressed RLE → convert to compressed first
            rle = mask_utils.frPyObjects(rle, height, width)
        return (mask_utils.decode(rle) * 255).astype(np.uint8)
    for poly in segmentation:  # list of polygons [[x1,y1,x2,y2,...], ...]
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        cv2.fillPoly(mask, [np.round(pts).astype(np.int32)], 255)
    return mask


# ============================================================ COCO parsing
def load_coco_records(data_dir: str, split: str) -> Tuple[List[dict], List[str]]:
    """
    Read a split folder ("train"/"valid"/"test") containing ``_annotations.coco.json``.

    Returns ``(records, class_names)`` where each record is a dict with
    ``image_path / segmentation / width / height / class_name / label``.

    - label = index from **sorted class names** (stable regardless of
      category_id ordering in the json).
    - 1 annotation = 1 sample → if one image contains multiple instruments
      it yields multiple samples (recommend using bbox_margin > 0 in
      Dataset to crop per instance).
    """
    split_dir = os.path.join(data_dir, split)
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    if not os.path.exists(ann_path):
        raise FileNotFoundError(f"Annotation file not found: {ann_path}")
    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = {im["id"]: im for im in coco["images"]}
    cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    # Filter to categories that actually appear (skip dummy super-category like id 0)
    used_names = {cat_id_to_name[ann["category_id"]] for ann in coco["annotations"]}
    class_names = sorted(used_names)
    name_to_label = {n: i for i, n in enumerate(class_names)}

    records: List[dict] = []
    for ann in coco["annotations"]:
        im = images[ann["image_id"]]
        cname = cat_id_to_name[ann["category_id"]]
        records.append({
            "image_path": os.path.join(split_dir, im["file_name"]),
            "segmentation": ann["segmentation"],
            "width": int(im["width"]),
            "height": int(im["height"]),
            "class_name": cname,
            "label": name_to_label[cname],
        })
    return records, class_names


def segmentation_bbox(segmentation, width: int, height: int) -> Tuple[int, int, int, int]:
    """Bounding box (x1,y1,x2,y2) enclosing all polygons — used when cropping per instance (bbox_margin > 0)."""
    xs: List[float] = []
    ys: List[float] = []
    for poly in segmentation if isinstance(segmentation, list) else []:
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        xs += [float(pts[:, 0].min()), float(pts[:, 0].max())]
        ys += [float(pts[:, 1].min()), float(pts[:, 1].max())]
    if not xs:  # RLE or empty polygon → use full image
        return 0, 0, width, height
    return int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))


# ============================================================ Augmentation
def build_photometric_aug() -> A.Compose:
    """
    Photometric augmentation for training — intentionally stronger than usual
    because metal reflections vary per capture, and the green cloth background
    with shifting shadows makes illumination inconsistent.

    Critically, there is *no* crop/scale/cutout because "size and shape"
    are what the model must learn to separate visually similar classes.
    """
    return A.Compose([
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),             # boost local contrast (metal on green cloth has low contrast; shadows worsen it)
        A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.7),  # stronger-than-usual jitter
        A.RandomGamma(gamma_limit=(70, 150), p=0.7),                        # simulate different exposure / lighting
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=15, p=0.3),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),                           # slight defocus blur
    ])


def build_tensor_transform(img_size: int) -> A.Compose:
    """Fixed-size resize + ImageNet normalization + conversion to tensor (used for train/eval/infer)."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def scharr_edge_map(gray: np.ndarray) -> np.ndarray:
    """
    Compute Scharr edge magnitude from a grayscale image (uint8) — returns float32 in [0, 1].
    Used for the SEF branch: emphasizes edges/shape of the instrument over color.
    """
    gx = cv2.Scharr(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Scharr(gray, cv2.CV_32F, 0, 1)
    mag = cv2.magnitude(gx, gy)
    m = float(mag.max())
    if m > 1e-6:
        mag = mag / m
    return mag.astype(np.float32)


def compute_class_length_means(records: List[dict], calibration_ratio: Optional[float] = None) -> dict:
    """Mean length per class (for LGMS) — use training records only."""
    from collections import defaultdict
    sums: dict = defaultdict(list)
    for r in records:
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        L = measure_length_px(mask)
        if calibration_ratio is not None:
            L *= calibration_ratio
        sums[r["label"]].append(L)
    return {k: float(np.mean(v)) for k, v in sums.items()}


def compute_lgms_margins(class_means: dict, num_classes: int, m_base: float = 28.6,
                         gamma: float = 10.0, k: int = 2) -> List[float]:
    """
    Compute per-class margin for LGMS.
    sim_len(i,j) = 1 - |len_i - len_j| / max_diff
    m(y) = m_base + gamma * mean(sim of k nearest neighbors)
    """
    means = np.array([class_means.get(i, 0.0) for i in range(num_classes)], dtype=np.float64)
    if num_classes <= 1:
        return [m_base] * num_classes
    diff = np.abs(means[:, None] - means[None, :])  # (C,C)
    max_diff = float(diff.max())
    if max_diff < 1e-6:
        return [m_base] * num_classes
    sim = 1.0 - diff / max_diff  # 0..1, higher = more similar length
    np.fill_diagonal(sim, -1)  # exclude self
    margins = []
    for y in range(num_classes):
        # top-k most similar
        topk_idx = np.argsort(sim[y])[::-1][:k]
        # filter negative values (when k > C-1)
        vals = [sim[y, j] for j in topk_idx if sim[y, j] >= 0]
        mean_sim = float(np.mean(vals)) if vals else 0.0
        margins.append(float(m_base + gamma * mean_sim))
    return margins

def simulate_shadow(img: np.ndarray, rng: Optional[random.Random] = None) -> np.ndarray:
    """
    Simulate "shadow" on the green cloth background — shadows shift with
    instrument placement / light direction.

    Draws 1-2 soft-edged dark blobs (ellipse + Gaussian falloff) multiplied
    onto the image. Implemented in numpy instead of A.RandomShadow because
    albumentations' signature changes frequently between 1.x ↔ 2.x — avoids
    version coupling.

    Green cloth shadows are the main difficulty for bounding/segmentation
    (not reflections on a silver tray).
    """
    rng = rng or random
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    mask = np.ones((h, w), dtype=np.float32)
    for _ in range(rng.randint(1, 3)):
        cx, cy = rng.uniform(0, w), rng.uniform(0, h)
        ax = rng.uniform(w * 0.2, w * 0.7)
        ay = rng.uniform(h * 0.2, h * 0.7)
        strength = rng.uniform(0.35, 0.65)          # darkest shadow ~35-65%
        d2 = ((xx - cx) / ax) ** 2 + ((yy - cy) / ay) ** 2
        mask *= 1.0 - strength * np.exp(-d2)
    out = img.astype(np.float32) * mask[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)


# ============================================================ Length statistics + split
def record_lengths(records: List[dict], calibration_ratio: Optional[float]) -> List[float]:
    """Measure length of every record (px or cm if ratio given) — rasterize directly from polygon."""
    out = []
    for r in records:
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        L = measure_length_px(mask)
        out.append(L if calibration_ratio is None else L * calibration_ratio)
    return out


def compute_length_stats(records: List[dict], calibration_ratio: Optional[float] = None) -> Tuple[float, float]:
    """
    Compute mean/std of lengths — **must be computed from train only**
    and the same values reused to normalize val/test/inference (avoid data leakage).
    """
    L = np.asarray(record_lengths(records, calibration_ratio), dtype=np.float64)
    mean = float(L.mean())
    std = max(float(L.std()), 1e-6)
    return mean, std


def stratified_split(records: List[dict], val_fraction: float = 0.2, seed: int = 42):
    """Stratified train/val split (preserve class proportions) — needed when data is scarce (~30 images/class)."""
    if val_fraction <= 0 or len(records) < 10:
        return records, []
    from sklearn.model_selection import train_test_split
    y = [r["label"] for r in records]
    tr, va = train_test_split(records, test_size=val_fraction, random_state=seed, stratify=y)
    return tr, va


# ============================================================ PyTorch Dataset
class SurgicalInstrumentDataset(Dataset):
    """
    Dataset for surgical instrument classification — returns a dict:
      ``image``  : FloatTensor (3, H, W) normalized
      ``length`` : scalar float = (length − mean) / std  ← auxiliary feature
      ``label``  : int64 class index

    Important notes:
    - Length is measured from the *original* mask (before augmentation) because
      photometric ops / flip should not change the true physical length.
    - ``flip_flags[label]`` must be True for that class to receive horizontal flip.
    - ``bbox_margin`` > 0 when one image contains multiple instruments → crop
      around the bbox of that instance (preserves aspect/scale, not a free zoom).
      Experiments found bbox_margin ≈ 0.15 effective; image size 504 was the
      best-performing default in experiments (this class defaults to 224 for
      backward compatibility — pass 504 explicitly to reproduce those results).
    - Background is green cloth; shadows on the cloth make tight bounding harder,
      which is why photometric + shadow augmentation is used.
    """

    def __init__(self, records: List[dict], length_stats: Tuple[float, float],
                 img_size: int = 224, calibration_ratio: Optional[float] = None,
                 flip_flags: Optional[List[bool]] = None, training: bool = True,
                 bbox_margin: float = 0.0, use_sef: bool = False):
        self.records = records
        self.length_mean, self.length_std = length_stats
        self.training = training
        self.bbox_margin = bbox_margin
        self.img_size = img_size
        self.use_sef = use_sef
        self.tensor_tf = build_tensor_transform(img_size)
        self.aug = build_photometric_aug() if training else None
        self.flip_flags = flip_flags
        # Measure length once at dataset creation (rasterizing polygons in memory is very fast)
        self._lengths = record_lengths(records, calibration_ratio)

    def __len__(self) -> int:
        return len(self.records)

    def _maybe_crop(self, img: np.ndarray, r: dict) -> np.ndarray:
        m = self.bbox_margin
        x1, y1, x2, y2 = segmentation_bbox(r["segmentation"], r["width"], r["height"])
        dx, dy = int((x2 - x1) * m), int((y2 - y1) * m)
        x1, y1 = max(x1 - dx, 0), max(y1 - dy, 0)
        x2, y2 = min(x2 + dx, r["width"]), min(y2 + dy, r["height"])
        return img[y1:y2, x1:x2]

    def __getitem__(self, idx: int) -> dict:
        r = self.records[idx]
        img = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            raise IOError(f"Failed to read image: {r['image_path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.bbox_margin > 0:
            img = self._maybe_crop(img, r)
        if self.training:
            if random.random() < 0.5:               # green-cloth shadow (real difficulty: lighting/shadow is the main challenge, not a silver tray)
                img = simulate_shadow(img)
            img = self.aug(image=img)["image"]
            # per-class horizontal flip (only for classes without handedness issues)
            if self.flip_flags is not None and self.flip_flags[r["label"]] and random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :])

        # main tensor
        tensor = self.tensor_tf(image=img)["image"]
        length_norm = (self._lengths[idx] - self.length_mean) / self.length_std
        out = {
            "image": tensor,
            "length": torch.tensor(length_norm, dtype=torch.float32),
            "label": torch.tensor(r["label"], dtype=torch.long),
        }
        # SEF: compute Scharr edge map from the augmented/flipped image then resize to img_size
        if self.use_sef:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            edge = scharr_edge_map(gray)  # (H,W) float [0,1]
            edge_resized = cv2.resize(edge, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
            out["edge_map"] = torch.from_numpy(edge_resized).unsqueeze(0).float()  # (1,H,W)
        return out


# ============================================================ Visualization (debug)
def visualize_records(records: List[dict], calibration_ratio: Optional[float] = None,
                      n: int = 6, cols: int = 3, seed: int = 0):
    """
    Display image + mask outline + measured length — use to verify that
    annotation / length measurement is correct before real training
    (returns a matplotlib figure).
    """
    import matplotlib.pyplot as plt
    rng = random.Random(seed)
    picks = rng.sample(records, min(n, len(records)))
    rows = math.ceil(len(picks) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.6 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(picks):]:
        ax.axis("off")
    unit = "cm" if calibration_ratio else "px"
    for ax, r in zip(axes, picks):
        bgr = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        mask = mask_from_coco_segmentation(r["segmentation"], r["height"], r["width"])
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, cnts, -1, (255, 40, 40), 3)
        L = measure_length_px(mask) * (calibration_ratio if calibration_ratio else 1.0)
        ax.imshow(img)
        ax.set_title(f"{r['class_name']} | {L:.1f} {unit}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    return fig


In [ ]:
%%writefile edge_branch.py
# -*- coding: utf-8 -*-
"""
edge_branch.py — Scharr Edge Fusion branch (SEF)

Small branch for processing edge maps computed from grayscale images using a Scharr filter,
then fused with the main embedding (DINOv2 + length) to emphasize shape/edges.

Architecture: 3x Conv2d(3x3) + BN + ReLU + GAP -> 64-dim vector
"""
import torch
import torch.nn as nn


class ScharrEdgeBranch(nn.Module):
    """
    Takes edge_map (B, 1, H, W) with float values in [0,1] — returns vector (B, 64)
    Used together with DeepFusionHead: concat(384 + 1 + 64) -> 384
    """
    def __init__(self, out_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Dropout(dropout),
        )
        self.out_dim = out_dim

    def forward(self, edge_map: torch.Tensor) -> torch.Tensor:
        """
        edge_map: (B, 1, H, W) — H, W may vary (GAP reduces to 1x1)
        return: (B, 64)
        """
        x = self.net(edge_map)  # (B, 64, 1, 1)
        return x.flatten(1)     # (B, 64)


In [ ]:
%%writefile model.py
# -*- coding: utf-8 -*-
"""
model.py — DINOv2 backbone + fusion head that fuses "length from mask" into the embedding

Architecture (v2 — improved):
    image -> DINOv2 ViT-S/14 -> all tokens (257 tokens, 384-dim)
    -> AttentionPooling: multi-head attention over patch tokens only -> weighted sum -> 384-dim
    mask -> measure_length_px() -> normalize(mean,std of train) -> scalar (1-dim)
    concat(384+1) -> DeepFusionHead (LN->Linear->GELU->Dropout->Linear->GELU->Dropout) -> 384-dim
    -> ArcFace loss (L2-normalize both embedding and class weights)

    Background: green cloth with shadows (not silver tray) — shadows make bounding-box
    detection harder; image size 504 and bbox 0.15 are defaults chosen from experiments.

v2 changes:
  - AttentionPooling instead of CLS-only: retains spatial detail from all patch tokens
  - DeepFusionHead: 2-layer MLP with LayerNorm for deeper fusion
  - Supports mask_aux_features: width/height/area/ratio from mask (optional, adds auxiliary info)

Why fuse length: resizing the image to 224x224 loses the true "real scale" information
but some class pairs differ only by length — so the length measured from the mask is fed in
directly to help.
"""
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import Dinov2Model

try:
    from edge_branch import ScharrEdgeBranch
    _EDGE_AVAILABLE = True
except ImportError:
    ScharrEdgeBranch = None  # type: ignore
    _EDGE_AVAILABLE = False

try:
    from peft import LoraConfig, TaskType, get_peft_model
    _PEFT_AVAILABLE = True
    _PEFT_IMPORT_ERROR = ""
except ImportError as e:
    # Colab has torchao 0.10.0 but peft 0.17+ requires >=0.16.0
    # Fall back to non-LoRA modes; user can fix via: !pip install -U torchao  (then restart runtime)
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)
except Exception as e:
    _PEFT_AVAILABLE = False
    _PEFT_IMPORT_ERROR = str(e)


class AttentionPooling(nn.Module):
    """
    Multi-head attention pooling over ViT patch tokens

    Instead of using the CLS token alone — learn attention weights
    over patch tokens only (excluding CLS) to retain spatial detail
    needed for fine-grained classification
    """

    def __init__(self, embed_dim: int, num_heads: int = 6, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)
        # learnable query — 1 token that attends to all patch tokens
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, num_tokens, embed_dim) — tokens from ViT (including CLS)
        return: (B, embed_dim) — pooled embedding
        """
        B = x.shape[0]
        # Separate patch tokens (index 1:) from CLS (index 0)
        patch_tokens = x[:, 1:, :]   # (B, 256, 384)
        q = self.query.expand(B, -1, -1)  # (B, 1, 384)
        attn_out, _ = self.attn(q, patch_tokens, patch_tokens)  # (B, 1, 384)
        attn_out = attn_out.squeeze(1)   # (B, 384)
        return self.norm(attn_out)


class DeepFusionHead(nn.Module):
    """
    Deep fusion head: concat visual embedding + length scalar -> project back to 384-dim

    v1: Linear(385->384) single layer — fast but shallow fusion
    v2: LN -> Linear(385->768) -> GELU -> Dropout -> Linear(768->384) -> Dropout
    """

    def __init__(self, embed_dim: int, aux_dim: int = 1, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(embed_dim + aux_dim)
        self.net = nn.Sequential(
            nn.Linear(embed_dim + aux_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, emb: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        """
        emb: (B, embed_dim) | aux: (B, aux_dim)
        return: (B, embed_dim)
        """
        x = torch.cat([emb, aux.to(emb.dtype)], dim=1)  # (B, embed_dim+aux_dim)
        x = self.ln(x)
        return self.net(x)


class SurgicalDinoFusion(nn.Module):
    """
    DINOv2 backbone (+LoRA/partial unfreeze) + AttentionPooling + DeepFusionHead

    finetune_mode:
      - "lora":    Wrap backbone with LoRA (train only adapters, very few parameters)
      - "partial": Freeze entire backbone then unfreeze last N blocks + final LayerNorm
      - "frozen":  Freeze entire backbone (use as feature extractor only)
    """

    def __init__(self,
                 backbone_name: str = "facebook/dinov2-small",
                 finetune_mode: str = "lora",
                 lora_r: int = 8,
                 lora_alpha: int = 16,
                 lora_dropout: float = 0.1,
                 partial_last_blocks: int = 2,
                 head_dropout: float = 0.1,
                 use_attention_pool: bool = True,
                 use_sef: bool = False,
                 sef_out_dim: int = 64):
        super().__init__()
        assert finetune_mode in ("frozen", "partial", "lora"), f"invalid mode: {finetune_mode}"

        self.backbone = Dinov2Model.from_pretrained(backbone_name)
        self.embed_dim = self.backbone.config.hidden_size  # 384 for dinov2-small
        self.use_attention_pool = use_attention_pool
        self.use_sef = use_sef

        if finetune_mode == "lora":
            if not _PEFT_AVAILABLE:
                hint = f" (detail: {_PEFT_IMPORT_ERROR[:120]})" if _PEFT_IMPORT_ERROR else ""
                raise ImportError(
                    "finetune_mode='lora' requires `peft` but import failed" + hint +
                    ". Fix in Colab: !pip install -U torchao  then Runtime -> Restart session, "
                    "or use finetune_mode='frozen'/'partial' to avoid LoRA."
                )
            lora_cfg = LoraConfig(
                task_type=TaskType.FEATURE_EXTRACTION,
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                target_modules=["query", "value"],  # LoRA only on attention query/value projections
                bias="none",
            )
            self.backbone = get_peft_model(self.backbone, lora_cfg)
        elif finetune_mode == "partial":
            self._freeze_partial(partial_last_blocks)
        else:  # frozen
            for p in self.backbone.parameters():
                p.requires_grad = False

        # Attention pooling: aggregate patch tokens -> 384-dim
        self.attn_pool = AttentionPooling(self.embed_dim, num_heads=6, dropout=head_dropout) if use_attention_pool else None
        # SEF branch (if enabled)
        self.edge_branch = None
        if use_sef:
            if not _EDGE_AVAILABLE:
                raise ImportError("use_sef=True requires edge_branch.py")
            self.edge_branch = ScharrEdgeBranch(out_dim=sef_out_dim, dropout=head_dropout)
        # Deep fusion head: concat(384 + 1 + 64_if_sef) -> 384
        aux_dim = 1 + (sef_out_dim if use_sef else 0)
        self.fusion = DeepFusionHead(self.embed_dim, aux_dim=aux_dim, dropout=head_dropout)

    def _freeze_partial(self, k: int) -> None:
        """Freeze entire backbone then unfreeze only last k blocks + final layernorm"""
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.encoder.layer[-k:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.layernorm.parameters():
            p.requires_grad = True

    def forward(self, pixel_values: torch.Tensor, length_feat: torch.Tensor,
                edge_map: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        pixel_values: (B,3,H,W) normalized | length_feat: (B,) normalized length
        edge_map: (B,1,H,W) Scharr edge [0,1] if use_sef=True — if None, zeros are used instead
        return: embedding (B, embed_dim=384) for ArcFace loss
        """
        out = self.backbone(pixel_values=pixel_values).last_hidden_state  # (B, tokens, 384)
        # Attention pooling: attend only to patch tokens instead of CLS-only
        if self.attn_pool is not None:
            e = self.attn_pool(out)        # (B, 384)
        else:
            e = out[:, 0]                   # CLS token fallback
        # Fusion: concat visual embedding + length scalar (+ 64 edge features if SEF)
        if self.use_sef and self.edge_branch is not None and edge_map is not None:
            ef = self.edge_branch(edge_map)  # (B, 64)
            aux = torch.cat([length_feat.unsqueeze(1), ef], dim=1)  # (B, 65)
        else:
            aux = length_feat.unsqueeze(1)      # (B, 1)
        return self.fusion(e, aux)          # (B, 384)

    def param_groups(self, lr_head: float, lr_backbone: Optional[float] = None) -> List[dict]:
        """
        Split parameter groups for differential learning rates:
          - head (attention_pool + fusion + edge_branch): lr_head
          - backbone where requires_grad=True (LoRA adapter or unfrozen blocks): lr_backbone
        """
        head_params = []
        if self.attn_pool is not None:
            head_params += list(self.attn_pool.parameters())
        head_params += list(self.fusion.parameters())
        if self.edge_branch is not None:
            head_params += list(self.edge_branch.parameters())
        groups = [{"params": [p for p in head_params if p.requires_grad], "lr": lr_head}]
        bb_trainable = [p for p in self.backbone.parameters() if p.requires_grad]
        if bb_trainable:
            groups.append({"params": bb_trainable, "lr": lr_backbone if lr_backbone is not None else lr_head})
        return groups

def arcface_logits(loss_fn, embeddings: torch.Tensor) -> torch.Tensor:
    """
    Logits at evaluate/inference: ``s · cos(θ)`` (no margin — margin is used only during training)

    Uses ``loss_fn.get_cosine()`` from pytorch-metric-learning directly — CosineSimilarity
    of the library normalizes both embedding and weight (W stored as shape (emb_dim, num_classes))
    itself, so it is correct on the "angle" for every version of the library
    """
    cos = loss_fn.get_cosine(embeddings)  # (B, num_classes), cosine of angle between vectors
    return cos * loss_fn.scale


class AdaptiveArcFaceLoss(torch.nn.Module):
    """
    Per-class margin ArcFace (for LGMS)

    margin_per_class: list/array of size num_classes in degrees
    Each sample uses the margin of its own label
    """
    def __init__(self, num_classes: int, embedding_size: int,
                 margin_per_class: List[float], scale: float = 64.0):
        super().__init__()
        from pytorch_metric_learning.distances import CosineSimilarity
        from pytorch_metric_learning.utils import common_functions as c_f
        self.num_classes = num_classes
        self.embedding_size = embedding_size
        self.scale = scale
        self.margin_per_class = np.array(margin_per_class, dtype=np.float64)
        self.margins_rad = np.radians(self.margin_per_class)
        self.W = torch.nn.Parameter(torch.Tensor(embedding_size, num_classes))
        # Xavier init like pytorch-metric-learning
        torch.nn.init.xavier_uniform_(self.W)
        self.cross_entropy = torch.nn.CrossEntropyLoss(reduction="none")
        self._cosine = CosineSimilarity()
        self._c_f = c_f

    def get_cosine(self, embeddings: torch.Tensor) -> torch.Tensor:
        # Follows pytorch-metric-learning: normalize embeddings and W
        emb_norm = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        W_norm = torch.nn.functional.normalize(self.W, p=2, dim=0)
        return torch.mm(emb_norm, W_norm)

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """
        Return mean loss — call compute_loss per-sample then mean
        so CAHM can weight per-sample afterwards (for per-sample loss call compute_loss_dict)
        """
        d = self.compute_loss_dict(embeddings, labels)
        return d["losses"].mean()

    def compute_loss_dict(self, embeddings: torch.Tensor, labels: torch.Tensor) -> dict:
        """
        Return dict {"losses": (B,) per-sample, "logits": (B,C)}
        for CAHM that needs per-sample weighting
        """
        dtype, device = embeddings.dtype, embeddings.device
        # Move W/margins to matching device/dtype
        W = self._c_f.to_device(self.W, device=device, dtype=dtype)
        margins = torch.as_tensor(self.margins_rad, device=device, dtype=dtype)  # (C,)
        # cosine (B, C)
        cosine = self.get_cosine(embeddings)  # internal uses self.W — need to ensure uses moved W; get_cosine uses normalized W directly
        # But get_cosine looks at self.W itself — cannot override to use moved W; do manual:
        emb_norm = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        W_norm = torch.nn.functional.normalize(W, p=2, dim=0)
        cosine = torch.mm(emb_norm, W_norm)
        # mask
        B = labels.size(0)
        mask = torch.zeros(B, self.num_classes, dtype=dtype, device=device)
        mask[torch.arange(B, device=device), labels] = 1
        cosine_target = cosine[mask == 1]  # (B,)
        angles = torch.acos(torch.clamp(cosine_target, -1 + 1e-7, 1 - 1e-7))
        m_per_sample = margins[labels]  # (B,)
        # cos(theta + m) as in ArcFace
        cos_theta_plus_m = torch.cos(angles + m_per_sample)
        cos_theta = torch.cos(angles)
        # keep monotonically decreasing (same as ArcFace)
        # if theta + m > pi fall back
        cond = angles <= (np.pi - m_per_sample)
        # m in radians must be converted to tensor for sin
        modified = torch.where(cond, cos_theta_plus_m, cos_theta - m_per_sample * torch.sin(torch.as_tensor(m_per_sample)))
        diff = (modified - cosine_target).unsqueeze(1)  # (B,1)
        logits = cosine + (mask * diff)
        logits = logits * self.scale
        losses = self.cross_entropy(logits, labels)  # (B,)
        return {"losses": losses, "logits": logits, "cosine": cosine}

    # Like pytorch-metric-learning: has attributes W and scale for arcface_logits
    @property
    def W_t(self):
        return self.W.t()


def count_trainable(model: nn.Module) -> int:
    """Count number of trainable parameters (used to verify LoRA/frozen is working correctly)"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile train.py
# -*- coding: utf-8 -*-
"""
train.py — training loop for DINOv2 + length fusion + ArcFace

Dataset context: instruments on green cloth background — shadows make tight
bounding challenging (not silver tray). Defaults img_size=504 and
bbox_margin=0.15 come from experiments (must be divisible by 14 for ViT patch size).

Usage from notebook/script:
    from config import TrainConfig
    from train import run_training
    best_ckpt = run_training(TrainConfig(data_dir="/content/dataset"))

Or via CLI:
    python train.py --data_dir dataset --epochs 50 --finetune_mode lora
    python train.py --data_dir dataset --kfold 5     # Stratified k-fold CV
"""
import argparse
import math
import os
import random
from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

from pytorch_metric_learning.losses import ArcFaceLoss

from config import TrainConfig
from dataset import (SurgicalInstrumentDataset, compute_class_length_means,
                     compute_length_stats, compute_lgms_margins,
                     load_coco_records, stratified_split)
from model import AdaptiveArcFaceLoss, SurgicalDinoFusion, arcface_logits, count_trainable


# ============================================================ utils
def seed_everything(seed: int) -> None:
    """Fix seeds for all RNGs to ensure reproducibility (critical with small data — different splits change results)"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4):
    """
    Mixup: random interpolation between randomly paired samples
    x: image tensor (B,3,H,W) | y: label (B,)
    return: mixed_x, y_a, y_b, lam (lambda = interpolation ratio)

    Important for small datasets: helps regularization by "blending" between classes
    alpha=0.4 → lam ~ Beta(0.4, 0.4) usually near 0 or 1 (not in the middle)
    """
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1.0 - lam)  # ensure lam >= 0.5 so labels don't swap
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def torch_load_compat(path: str) -> dict:
    """torch.load compatible with both old and new torch (default weights_only changed in torch 2.6)"""
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def build_flip_flags(class_names: List[str], cfg: TrainConfig) -> List[bool]:
    """Only classes listed in cfg.flip_allowed can be flipped (None = all classes can be flipped)"""
    allowed = set(cfg.flip_allowed) if cfg.flip_allowed is not None else None
    return [True if allowed is None else n in allowed for n in class_names]


def resolve_records(cfg: TrainConfig) -> Tuple[List[dict], List[dict], List[str]]:
    """
    Load records from data_dir:
      - if valid/_annotations.coco.json exists → use it directly
      - otherwise → stratified split from train using val_fraction/seed from config
    """
    tr, classes = load_coco_records(cfg.data_dir, "train")
    valid_ann = os.path.join(cfg.data_dir, "valid", "_annotations.coco.json")
    if os.path.exists(valid_ann):
        va, classes_valid = load_coco_records(cfg.data_dir, "valid")
        if classes_valid != classes:
            raise ValueError(f"Class lists differ between train/valid:\n{classes}\n{classes_valid}")
    else:
        tr, va = stratified_split(tr, cfg.val_fraction, cfg.seed)
        print(f"[data] no valid/ folder → stratified split {len(tr)}/{len(va)} (seed={cfg.seed})")
    return tr, va, classes

def warmup_cosine_factor(step: int, warmup: int, total: int) -> float:
    """LR schedule: linear warmup → cosine decay to near 0 by the end"""
    if step < warmup:
        return step / max(1, warmup)
    t = min((step - warmup) / max(1, total - warmup), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * t))


# ============================================================ CAHM helpers
def _cahm_d_from_cm(cm: np.ndarray) -> np.ndarray:
    """
    Compute pair difficulty from confusion matrix:
      d(i,j) = C[i,j] + C[j,i]  (i!=j), normalized by max
    """
    C = cm.astype(np.float64)
    n = C.shape[0]
    d = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(n):
            if i != j:
                d[i, j] = C[i, j] + C[j, i]
    m = d.max()
    if m > 1e-9:
        d = d / m
    return d


def _cahm_weights(labels: torch.Tensor, d_t: np.ndarray, alpha: float, device) -> torch.Tensor:
    """w = 1 + alpha * max_j d_t[y,j]  (per-sample)"""
    if d_t is None:
        return torch.ones_like(labels, dtype=torch.float32)
    d = torch.as_tensor(d_t, device=device, dtype=torch.float32)  # (C,C)
    # row-wise max (diagonal is already 0, so no need to exclude)
    row_max = d.max(dim=1).values  # (C,)
    w = 1.0 + alpha * row_max[labels]
    return w


@torch.no_grad()
def _eval_confusion(model, loss_fn, loader, device, num_classes: int) -> np.ndarray:
    """Run over the full validation set to build a confusion matrix for CAHM"""
    from sklearn.metrics import confusion_matrix
    model.eval()
    ys_true, ys_pred = [], []
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device, non_blocking=True)
        emb = model(px, ln, edge)
        logits = arcface_logits(loss_fn, emb.float())
        pred = logits.argmax(dim=1).cpu().numpy()
        ys_true.extend(y.cpu().numpy().tolist())
        ys_pred.extend(pred.tolist())
    cm = confusion_matrix(ys_true, ys_pred, labels=list(range(num_classes)))
    return cm
# ============================================================ epochs
def train_one_epoch(model, loss_fn, loader, optimizer, scheduler, scaler, device, cfg,
                    cahm_d: Optional[np.ndarray] = None) -> float:
    """Train for 1 epoch → return average loss (ArcFace on embeddings from the fusion head)"""
    model.train()
    total, seen = 0.0, 0
    mixup_alpha = getattr(cfg, "mixup_alpha", 0.0)
    use_cahm = bool(getattr(cfg, "use_cahm", False)) and cahm_d is not None
    cahm_alpha = float(getattr(cfg, "cahm_alpha", 2.0))
    is_adaptive = hasattr(loss_fn, "compute_loss_dict")
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device, non_blocking=True)

        # Mixup: randomly interpolate between 2 samples (regularization for small datasets)
        use_mixup = mixup_alpha > 0 and model.training and not use_cahm  # disable mixup when using CAHM so weights remain clear
        if use_mixup:
            px, y_a, y_b, lam = mixup_data(px, y, mixup_alpha)
        else:
            y_a, y_b, lam = y, y, 1.0

        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:  # GPU → mixed precision
            with torch.autocast("cuda"):
                emb = model(px, ln, edge)
                if use_mixup:
                    loss = lam * loss_fn(emb.float(), y_a) + (1 - lam) * loss_fn(emb.float(), y_b)
                elif use_cahm:
                    # CAHM weighted loss — retrieve per-sample loss and multiply by w
                    if is_adaptive:
                        d = loss_fn.compute_loss_dict(emb.float(), y)
                        per = d["losses"]  # (B,)
                    else:
                        # must pass ref_emb = embeddings to pass PML identity check
                        ef = emb.float()
                        ld = loss_fn.compute_loss(ef, y, None, ef, y)
                        per = ld["loss"]["losses"]  # (B,)
                    w = _cahm_weights(y, cahm_d, cahm_alpha, device)
                    loss = (per * w).mean()
                else:
                    loss = loss_fn(emb.float(), y)  # cast to fp32 before ArcFace for stability
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:  # CPU → fp32
            emb = model(px, ln, edge)
            if use_mixup:
                loss = lam * loss_fn(emb, y_a) + (1 - lam) * loss_fn(emb, y_b)
            elif use_cahm:
                if is_adaptive:
                    d = loss_fn.compute_loss_dict(emb.float(), y)
                    per = d["losses"]
                else:
                    ef = emb.float()
                    ld = loss_fn.compute_loss(ef, y, None, ef, y)
                    per = ld["loss"]["losses"]
                w = _cahm_weights(y, cahm_d, cahm_alpha, device)
                loss = (per * w).mean()
            else:
                loss = loss_fn(emb, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        scheduler.step()  # per-step schedule (warmup+cosine)

        total += loss.item() * y.size(0)
        seen += y.size(0)
    return total / max(seen, 1)


@torch.no_grad()
def validate(model, loss_fn, loader, device) -> Tuple[float, float]:
    """
    validation -> (val_loss, val_acc)
    - val_loss: ArcFace loss on the val set (logged; tiebreak for best-model selection)
    - val_acc : argmax over s*cos(theta) logits (true inference mode, no margin)
    """
    model.eval()
    embs, ys = [], []
    for batch in loader:
        px = batch["image"].to(device, non_blocking=True)
        ln = batch["length"].to(device, non_blocking=True)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device, non_blocking=True)
        embs.append(model(px, ln, edge).float().cpu())
        ys.append(batch["label"])
    E = torch.cat(embs).to(device)
    Y = torch.cat(ys).to(device)
    loss = loss_fn(E, Y).item()
    acc = (arcface_logits(loss_fn, E).argmax(dim=1) == Y).float().mean().item()
    return loss, acc

def save_history_plot(history: dict, out_png: str) -> None:
    """Plot loss/accuracy — may fail (headless) without crashing training"""
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].plot(history["train_loss"], label="train")
        ax[0].plot(history["val_loss"], label="val")
        ax[0].set_title("ArcFace loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
        ax[1].plot(history["val_acc"], color="tab:green")
        ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=.3)
        fig.savefig(out_png, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"[log] saved history plot → {out_png}")
    except Exception as e:  # noqa: BLE001 — plotting should not crash training
        print(f"(skipping history plot: {e})")


# ============================================================ main training entry
def run_training(cfg: TrainConfig,
                 records_train: Optional[List[dict]] = None,
                 records_valid: Optional[List[dict]] = None,
                 tag: str = "") -> str:
    """
    Train once (single split) — returns path of the best checkpoint (highest val accuracy)

    checkpoint contains: model_state, arcface_state, classes, length_mean/std,
                         cfg (dict), epoch, val_loss, val_acc
    """
    os.makedirs(cfg.output_dir, exist_ok=True)
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- data ----------
    if records_train is None or records_valid is None:
        records_train, records_valid, _ = resolve_records(cfg)
    all_recs = list(records_train) + list(records_valid)
    label2name = {r["label"]: r["class_name"] for r in all_recs}
    class_names = [label2name[i] for i in range(max(label2name) + 1)]  # indices always sorted

    # length mean/std ← from train only (prevent leakage)
    length_stats = compute_length_stats(records_train, cfg.calibration_ratio)
    print(f"[data] train={len(records_train)} val={len(records_valid)} "
          f"classes={len(class_names)} length_mean={length_stats[0]:.2f} std={length_stats[1]:.2f}")

    flip_flags = build_flip_flags(class_names, cfg)
    use_sef = bool(getattr(cfg, "use_sef", False))
    ds_train = SurgicalInstrumentDataset(records_train, length_stats, cfg.img_size,
                                         cfg.calibration_ratio, flip_flags, training=True,
                                         bbox_margin=cfg.bbox_margin, use_sef=use_sef)
    ds_val = SurgicalInstrumentDataset(records_valid, length_stats, cfg.img_size,
                                       cfg.calibration_ratio, flip_flags=None, training=False,
                                       bbox_margin=cfg.bbox_margin, use_sef=use_sef)
    pin = device.type == "cuda"
    dl_train = DataLoader(ds_train, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=pin)
    dl_val = DataLoader(ds_val, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=pin)
    # ---------- model + loss ----------
    model = SurgicalDinoFusion(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks, head_dropout=cfg.head_dropout,
        use_attention_pool=getattr(cfg, "use_attention_pool", True),
        use_sef=use_sef,
    ).to(device)
    print(f"[model] trainable params = {count_trainable(model):,} (mode={cfg.finetune_mode}, sef={use_sef})")

    # ArcFace — standard or LGMS (per-class margin)
    if bool(getattr(cfg, "use_lgms", False)):
        class_means = compute_class_length_means(records_train, cfg.calibration_ratio)
        margins = compute_lgms_margins(class_means, len(class_names),
                                       m_base=cfg.margin, gamma=cfg.lgms_gamma, k=cfg.lgms_k)
        print(f"[LGMS] margins per class: {[f'{m:.1f}' for m in margins]}")
        loss_fn = AdaptiveArcFaceLoss(num_classes=len(class_names), embedding_size=model.embed_dim,
                                      margin_per_class=margins, scale=cfg.scale).to(device)
    else:
        # ArcFace: margin in degrees (~28.6° = 0.5 rad), s=64 — forces embeddings to have
        # tight intra-class / wide inter-class separation, suitable for classes with very similar shapes
        loss_fn = ArcFaceLoss(num_classes=len(class_names), embedding_size=model.embed_dim,
                              margin=cfg.margin, scale=cfg.scale).to(device)
    if cfg.finetune_mode == "partial":
        lr_bb = cfg.lr_backbone
    elif cfg.finetune_mode == "lora":
        lr_bb = cfg.lr_lora
    else:
        lr_bb = None
    optimizer = AdamW(model.param_groups(cfg.lr_head, lr_bb), weight_decay=cfg.weight_decay)

    total_steps = max(1, len(dl_train)) * cfg.epochs
    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))
    scheduler = LambdaLR(optimizer, lr_lambda=lambda s: warmup_cosine_factor(s, warmup_steps, total_steps))

    if device.type == "cuda":
        try:
            scaler = torch.amp.GradScaler("cuda")   # torch >= 2.3
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler()    # fallback for old torch
    else:
        scaler = None

    # ---------- loop + early stopping ----------
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    # Select best by "val_acc" (val_loss as tiebreak) — simulations show that on small datasets
    # ArcFace val_loss and val_acc conflict (lowest loss != best model); original spec used val loss
    best = {"val_loss": float("inf"), "val_acc": -1.0, "epoch": -1, "model": None, "arcface": None}
    bad_epochs = 0
    cahm_d = None  # (C,C) EMA state for CAHM
    use_cahm = bool(getattr(cfg, "use_cahm", False))
    cahm_start = int(getattr(cfg, "cahm_start_epoch", 10))
    cahm_beta = float(getattr(cfg, "cahm_beta", 0.9))

    for epoch in range(1, cfg.epochs + 1):
        # pass cahm_d to this epoch if start time has been reached
        cur_cahm = cahm_d if (use_cahm and epoch > cahm_start and cahm_d is not None) else None
        tl = train_one_epoch(model, loss_fn, dl_train, optimizer, scheduler, scaler, device, cfg, cahm_d=cur_cahm)
        vl, va = validate(model, loss_fn, dl_val, device)
        history["train_loss"].append(tl); history["val_loss"].append(vl); history["val_acc"].append(va)

        # CAHM: update difficulty after validation (for next epoch)
        if use_cahm and epoch >= cahm_start:
            try:
                cm = _eval_confusion(model, loss_fn, dl_val, device, len(class_names))
                d_cur = _cahm_d_from_cm(cm)
                if cahm_d is None:
                    cahm_d = d_cur
                else:
                    cahm_d = cahm_beta * cahm_d + (1 - cahm_beta) * d_cur
                # log top-1 confused pair for debugging
                flat = [(i, j, cahm_d[i, j]) for i in range(len(class_names)) for j in range(len(class_names)) if i != j]
                flat.sort(key=lambda x: -x[2])
                if flat:
                    i, j, v = flat[0]
                    print(f"  [CAHM] top confused: {class_names[i]}↔{class_names[j]} d={v:.3f}")
            except Exception as e:
                print(f"  [CAHM] skip update: {e}")

        improved = va > best["val_acc"] + 1e-4 or \
            (va >= best["val_acc"] - 1e-4 and vl < best["val_loss"] - 1e-4)
        star = ""
        if improved:
            best.update(val_loss=vl, val_acc=va, epoch=epoch,
                        model={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                        arcface={k: v.detach().cpu().clone() for k, v in loss_fn.state_dict().items()})
            bad_epochs = 0
            star = "  *best*"
            # save immediately — checkpoint survives interruption (file exists even with early stop)
            try:
                ckpt_immediate = os.path.join(cfg.output_dir, f"best_model{tag}.pt")
                torch.save({
                    "model_state": best["model"],
                    "arcface_state": best["arcface"],
                    "classes": class_names,
                    "length_mean": float(length_stats[0]),
                    "length_std": float(length_stats[1]),
                    "calibration_ratio": cfg.calibration_ratio,
                    "cfg": cfg.to_dict(),
                    "epoch": best["epoch"],
                    "val_loss": float(best["val_loss"]),
                    "val_acc": float(best["val_acc"]),
                }, ckpt_immediate)
            except Exception as e:
                print(f"(skipping immediate save: {e})")
        else:
            bad_epochs += 1
        cur_lr = scheduler.get_last_lr()[0]
        print(f"{tag}[epoch {epoch:03d}/{cfg.epochs}] train={tl:.4f} val={vl:.4f} "
              f"val_acc={va:.4f} lr={cur_lr:.2e}{star}", flush=True)

        if bad_epochs >= cfg.patience:
            print(f"[early stop] no improvement for {cfg.patience} epochs — stopping at epoch {epoch}")
            break

    # ---------- save best checkpoint ----------
    ckpt_path = os.path.join(cfg.output_dir, f"best_model{tag}.pt")
    torch.save({
        "model_state": best["model"],
        "arcface_state": best["arcface"],
        "classes": class_names,
        "length_mean": float(length_stats[0]),
        "length_std": float(length_stats[1]),
        "calibration_ratio": cfg.calibration_ratio,
        "cfg": cfg.to_dict(),
        "epoch": best["epoch"],
        "val_loss": float(best["val_loss"]),
        "val_acc": float(best["val_acc"]),
    }, ckpt_path)
    save_history_plot(history, os.path.join(cfg.output_dir, f"history{tag}.png"))

    print(f"[done] best epoch={best['epoch']} val_loss={best['val_loss']:.4f} "
          f"val_acc={best['val_acc']:.4f} → {ckpt_path}")
    return ckpt_path


# ============================================================ k-fold CV
def run_kfold(cfg: TrainConfig, k: Optional[int] = None) -> Tuple[List[str], List[float]]:
    """
    Stratified k-fold CV over all data (train∪valid) — more reliable than single split
    when there are only ~30 images/class; returns (paths, val_acc per fold)
    """
    from sklearn.model_selection import StratifiedKFold
    tr, va, _ = resolve_records(cfg)
    all_recs = tr + va
    y = [r["label"] for r in all_recs]
    skf = StratifiedKFold(n_splits=k or cfg.kfold, shuffle=True, random_state=cfg.seed)

    paths, accs = [], []
    for i, (idx_tr, idx_va) in enumerate(skf.split(all_recs, y)):
        rec_tr = [all_recs[j] for j in idx_tr]
        rec_va = [all_recs[j] for j in idx_va]
        print(f"\n========== Fold {i + 1}/{skf.n_splits} "
              f"(train={len(rec_tr)} val={len(rec_va)}) ==========")
        path = run_training(cfg, rec_tr, rec_va, tag=f"_fold{i + 1}")
        paths.append(path)
        ckpt = torch_load_compat(path)
        accs.append(float(ckpt["val_acc"]))

    print("\n===== K-FOLD SUMMARY =====")
    for i, a in enumerate(accs):
        print(f"fold {i + 1}: val_acc={a:.4f}")
    print(f"mean={np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return paths, accs


# ============================================================ CLI
def main(argv=None) -> None:
    from dataclasses import replace
    ap = argparse.ArgumentParser(description="Train DINOv2+length-fusion+ArcFace for surgical instrument classification")
    ap.add_argument("--data_dir", default="dataset")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--batch_size", type=int, default=None)
    ap.add_argument("--img_size", type=int, default=None)
    ap.add_argument("--finetune_mode", choices=["lora", "partial", "frozen"], default=None)
    ap.add_argument("--kfold", type=int, default=None, help="e.g. 5 → Stratified 5-fold CV")
    ap.add_argument("--calibration_ratio", type=float, default=None,
                    help="cm/pixel from reference object (omit = use pixels)")
    ap.add_argument("--output_dir", default=None)
    ap.add_argument("--seed", type=int, default=None)
    # CAHM / LGMS / SEF — toggle auxiliary algorithms
    ap.add_argument("--use_cahm", action="store_true", help="enable CAHM (confusion-aware hard mining)")
    ap.add_argument("--use_lgms", action="store_true", help="enable LGMS (length-gated margin scaling)")
    ap.add_argument("--use_sef", action="store_true", help="enable SEF (Scharr edge fusion)")
    ap.add_argument("--cahm_alpha", type=float, default=None)
    ap.add_argument("--cahm_beta", type=float, default=None)
    ap.add_argument("--lgms_gamma", type=float, default=None)
    ap.add_argument("--lgms_k", type=int, default=None)
    args = ap.parse_args(argv)

    overrides = {}
    for k, v in vars(args).items():
        if k == "calibration_ratio":
            continue
        if v is None:
            continue
        # store_true flags: False means not set → don't override (keep default False)
        if isinstance(v, bool) and not v:
            continue
        overrides[k] = v
    if args.calibration_ratio is not None:
        overrides["calibration_ratio"] = args.calibration_ratio
    cfg = replace(TrainConfig(), **overrides)
    if cfg.kfold and cfg.kfold > 1:
        run_kfold(cfg)
    else:
        path = run_training(cfg)
        print("checkpoint:", path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
# -*- coding: utf-8 -*-
"""
evaluate.py — Evaluate a trained checkpoint

Usage from notebook/script:
    from evaluate import evaluate_checkpoint
    metrics = evaluate_checkpoint("outputs/best_model.pt")

Returns overall accuracy, per-class classification report, confusion matrix
(heatmap) and "most frequently confused class pairs" which are often pairs
differing only in size.
"""
import os
from dataclasses import replace
from typing import List, Optional

import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                             confusion_matrix)

from config import TrainConfig
from dataset import SurgicalInstrumentDataset
from model import SurgicalDinoFusion, arcface_logits
from pytorch_metric_learning.losses import ArcFaceLoss
from train import resolve_records, torch_load_compat



def load_bundle(ckpt_path: str, device: Optional[torch.device] = None) -> dict:
    """
    Load checkpoint -> build model + ArcFace head ready for inference
    (cfg is stored in the checkpoint at training time -> structure can be reproduced exactly)
    """
    ckpt = torch_load_compat(ckpt_path)
    cfg = TrainConfig(**ckpt["cfg"])
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Must be created with the original finetune_mode so that state_dict keys match
    model = SurgicalDinoFusion(
        backbone_name=cfg.backbone_name, finetune_mode=cfg.finetune_mode,
        lora_r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        partial_last_blocks=cfg.partial_last_blocks, head_dropout=cfg.head_dropout,
        use_attention_pool=getattr(cfg, "use_attention_pool", True),
        use_sef=bool(getattr(cfg, "use_sef", False)),
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(device).eval()

    classes: List[str] = ckpt["classes"]
    arcface = ArcFaceLoss(num_classes=len(classes), embedding_size=model.embed_dim,
                          margin=cfg.margin, scale=cfg.scale)
    try:
        arcface.load_state_dict(ckpt["arcface_state"])
    except Exception:
        # If trained with LGMS (AdaptiveArcFace) but evaluated with standard ArcFace — W can still be loaded
        # Try loading with strict=False
        arcface.load_state_dict(ckpt["arcface_state"], strict=False)
    arcface.to(device)

    return {"model": model, "arcface": arcface, "classes": classes, "cfg": cfg,
            "device": device,
            "length_mean": float(ckpt["length_mean"]), "length_std": float(ckpt["length_std"]),
            "calibration_ratio": ckpt.get("calibration_ratio")}

@torch.no_grad()
def predict_all(bundle: dict, records: List[dict]):
    """Run over the entire validation set -> (y_true, y_pred, confidence of the predicted class)"""
    cfg = bundle["cfg"]
    device = bundle["device"]
    length_stats = (bundle["length_mean"], bundle["length_std"])
    use_sef = bool(getattr(cfg, "use_sef", False))
    ds = SurgicalInstrumentDataset(records, length_stats, cfg.img_size,
                                   cfg.calibration_ratio, flip_flags=None, training=False,
                                   bbox_margin=getattr(cfg, "bbox_margin", 0.0),
                                   use_sef=use_sef)
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    y_true, y_pred, y_conf = [], [], []
    for batch in dl:
        px = batch["image"].to(device)
        ln = batch["length"].to(device)
        edge = batch.get("edge_map")
        if edge is not None:
            edge = edge.to(device)
        emb = bundle["model"](px, ln, edge)
        logits = arcface_logits(bundle["arcface"], emb.float())
        probs = torch.softmax(logits, dim=-1)
        conf, pred = probs.max(dim=-1)
        y_pred += pred.cpu().tolist()
        y_conf += conf.cpu().tolist()
        y_true += batch["label"].tolist()
    return np.array(y_true), np.array(y_pred), np.array(y_conf)


def plot_confusion_matrix(cm: np.ndarray, class_names: List[str],
                          save_path: Optional[str] = None, figsize=(12, 10)):
    """Heatmap of the confusion matrix (uses seaborn if available, otherwise pure matplotlib)"""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=figsize)
    try:
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt="d", cmap="viridis",
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
    except ImportError:
        im = ax.imshow(cm, cmap="viridis")
        fig.colorbar(im, ax=ax)
        ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=90)
        ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight")
        print(f"[log] Saved confusion matrix -> {save_path}")
    return fig


def print_top_confused(cm: np.ndarray, class_names: List[str], top_n: int = 10) -> None:
    """Print the most frequently confused class pairs (true -> predicted) — indicates what to fix next, e.g. adding size features"""
    pairs = [(int(cm[i, j]), i, j)
             for i in range(len(class_names)) for j in range(len(class_names))
             if i != j and cm[i, j] > 0]
    if not pairs:
        print("No cross-class confusion at all 🎉")
        return
    pairs.sort(reverse=True)
    print("\nMost confused class pairs (true -> predicted):")
    for cnt, i, j in pairs[:top_n]:
        print(f"  {class_names[i]} -> {class_names[j]} : {cnt} times")


def evaluate_checkpoint(ckpt_path: str, data_dir: Optional[str] = None,
                        show_plot: bool = True, save_dir: Optional[str] = None) -> dict:
    """
    Evaluate checkpoint on the validation set -> dict containing
      accuracy / balanced_accuracy / report / confusion_matrix / cm_path / fig
    """
    bundle = load_bundle(ckpt_path)
    cfg = bundle["cfg"]
    if data_dir:
        cfg = replace(cfg, data_dir=data_dir)
    _, va_records, _ = resolve_records(cfg)
    if len(va_records) == 0:
        raise ValueError("validation set is empty — check data_dir/val_fraction")

    y_true, y_pred, _ = predict_all(bundle, va_records)
    classes = bundle["classes"]
    acc = float((y_true == y_pred).mean())
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))

    present = sorted(set(y_true.tolist()))
    names_present = [classes[i] for i in present]
    print(f"\n===== Evaluation ({len(va_records)} samples) =====")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print("\n" + classification_report(
        [classes[i] for i in y_true], [classes[i] for i in y_pred],
        labels=names_present, digits=3, zero_division=0))

    out_dir = save_dir or cfg.output_dir
    os.makedirs(out_dir, exist_ok=True)
    cm_path = os.path.join(out_dir, "confusion_matrix.png")
    fig = plot_confusion_matrix(cm, classes, save_path=cm_path)
    if show_plot:
        try:
            import matplotlib.pyplot as plt
            plt.show()
        except Exception:
            pass
    print_top_confused(cm, classes)

    return {"accuracy": acc, "balanced_accuracy": bal_acc,
            "report": classification_report(
                [classes[i] for i in y_true], [classes[i] for i in y_pred],
                output_dict=True, zero_division=0),
            "confusion_matrix": cm, "cm_path": cm_path, "fig": fig}


In [ ]:
%%writefile infer.py
# -*- coding: utf-8 -*-
"""
infer.py — inference for a single new image (+mask) → class + confidence

Usage from notebook/script:
    from infer import load_pipeline, predict_record, predict_file
    pack = load_pipeline("outputs/best_model.pt")
    res  = predict_file(pack, image_path="x.jpg", mask_png="x_mask.png")
    # or with COCO json: predict_file(pack, "x.jpg", coco_json="_annotations.coco.json",
    #                                 image_filename="x.jpg")

CLI:
    python infer.py --ckpt outputs/best_model.pt --image x.jpg --mask_png x_mask.png

Note: confidence is softmax of s·cos(θ) — useful for comparing relative
confidence between classes, but not a true calibrated probability.
"""
import argparse
import json
import os
from typing import List, Optional

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from dataset import build_tensor_transform, mask_from_coco_segmentation, measure_length_px, scharr_edge_map
from model import arcface_logits


def load_pipeline(ckpt_path: str, device: Optional[torch.device] = None) -> dict:
    """Load checkpoint → pack (model, arcface head, transform, metadata)"""
    from evaluate import load_bundle
    pack = load_bundle(ckpt_path, device)
    pack["tensor_tf"] = build_tensor_transform(pack["cfg"].img_size)
    # Store cfg as dict so predict_array can read use_tta/use_sef
    if hasattr(pack["cfg"], "to_dict"):
        # Keep original object as well for img_size
        pack["_cfg_obj"] = pack["cfg"]
        pack["cfg"] = pack["cfg"].to_dict()
    return pack


def _predict_once(pack: dict, image_rgb: np.ndarray, ln_norm: float) -> torch.Tensor:
    """Single forward pass → returns logits tensor (num_classes,)"""
    tensor = pack["tensor_tf"](image=image_rgb)["image"][None].to(pack["device"])
    ln = torch.tensor([ln_norm], dtype=torch.float32, device=pack["device"])
    edge = None
    cfg = pack.get("cfg", {})
    if cfg.get("use_sef", False):
        # Scharr edge map from the image after crop/resize, same as used during training
        gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
        edge_np = scharr_edge_map(gray)
        h, w = image_rgb.shape[:2]
        # edge must be resized to match tensor (img_size)
        img_size = cfg.get("img_size", 616)
        edge_resized = cv2.resize(edge_np, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        edge = torch.from_numpy(edge_resized).unsqueeze(0).unsqueeze(0).to(pack["device"]).float()
    emb = pack["model"](tensor, ln, edge)
    return arcface_logits(pack["arcface"], emb.float())[0]

@torch.no_grad()
def predict_array(pack: dict, image_rgb: np.ndarray,
                  mask_gray: Optional[np.ndarray] = None) -> dict:
    """
    Predict 1 image with TTA (Test-Time Augmentation)
      image_rgb : uint8 (H,W,3) RGB
      mask_gray : binary mask of the instrument (H,W) — if not provided, the
                  training mean length is used instead (model can still predict
                  but is less accurate for class pairs that differ by size)

    TTA: original + horizontal flip → average logits → more stable than single prediction
    """
    ratio = pack.get("calibration_ratio")
    if mask_gray is not None:
        length = measure_length_px(mask_gray) * (ratio if ratio else 1.0)
    else:
        length = pack["length_mean"]  # neutral fallback

    ln = (length - pack["length_mean"]) / pack["length_std"]

    # TTA: original + horizontal flip → average logits
    use_tta = pack.get("cfg", {}).get("use_tta", False) if isinstance(pack.get("cfg"), dict) else False
    if use_tta:
        logits_orig = _predict_once(pack, image_rgb, ln)
        # horizontal flip of mask as well (if present) — no need to flip length, length is invariant
        img_flip = np.ascontiguousarray(image_rgb[:, ::-1, :])
        logits_flip = _predict_once(pack, img_flip, ln)
        logits = (logits_orig + logits_flip) / 2.0
    else:
        logits = _predict_once(pack, image_rgb, ln)

    probs = F.softmax(logits, dim=-1).cpu()

    top3 = probs.topk(3)
    classes: List[str] = pack["classes"]
    return {
        "class": classes[int(top3.indices[0])],
        "confidence": float(top3.values[0]),
        "top3": [{"class": classes[int(i)], "prob": float(p)}
                 for p, i in zip(top3.values, top3.indices)],
        "length_used": float(length),
        "tta_used": use_tta,
    }


def predict_record(pack: dict, record: dict) -> dict:
    """Predict from a record returned by load_coco_records() (uses segmentation polygon in record)"""
    bgr = cv2.imread(record["image_path"], cv2.IMREAD_COLOR)
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    mask = mask_from_coco_segmentation(record["segmentation"], record["height"], record["width"])
    res = predict_array(pack, img, mask)
    res["truth"] = record["class_name"]
    return res


def predict_file(pack: dict, image_path: str, mask_png: Optional[str] = None,
                 coco_json: Optional[str] = None,
                 image_filename: Optional[str] = None) -> dict:
    """
    Predict from files:
      - mask_png   : mask file (white/black) if available
      - coco_json  : or point to an annotation file and specify image_filename → uses the first polygon annotation for that image
    """
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise IOError(f"Failed to read image: {image_path}")
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    mask = None
    if mask_png:
        m = cv2.imread(mask_png, cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise IOError(f"Failed to read mask: {mask_png}")
        mask = ((m > 127).astype(np.uint8)) * 255
    elif coco_json:
        fname = image_filename or os.path.basename(image_path)
        with open(coco_json, "r", encoding="utf-8") as f:
            coco = json.load(f)
        images = {im["id"]: im for im in coco["images"]}
        target = next((im for im in coco["images"] if im["file_name"] == fname), None)
        if target is None:
            raise KeyError(f"{fname} not found in {coco_json}")
        ann = next((a for a in coco["annotations"] if a["image_id"] == target["id"]), None)
        if ann is None:
            raise KeyError(f"{fname} has no annotation")
        mask = mask_from_coco_segmentation(ann["segmentation"],
                                           int(images[target["id"]]["height"]),
                                           int(images[target["id"]]["width"]))
    return predict_array(pack, img, mask)


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description="Inference DINOv2+ArcFace")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--image", required=True)
    ap.add_argument("--mask_png", default=None)
    ap.add_argument("--coco_json", default=None)
    args = ap.parse_args(argv)

    pack = load_pipeline(args.ckpt)
    res = predict_file(pack, args.image, mask_png=args.mask_png, coco_json=args.coco_json)
    print(json.dumps(res, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
!mkdir -p tools

In [ ]:
%%writefile tools/evaluate_ablation.py
# -*- coding: utf-8 -*-
"""
tools/evaluate_ablation.py — วัดและเปรียบเทียบผล 6 configs สำหรับ Phase 3

ใช้บน Colab เป็นเซลล์เดียว:
    from tools.evaluate_ablation import compare_checkpoints
    table = compare_checkpoints({
        "baseline": "outputs/best_model.pt",
        "CAHM":     "outputs/best_model_cahm.pt",
        "LGMS":     "outputs/best_model_lgms.pt",
        "SEF":      "outputs/best_model_sef.pt",
        "CAHM+LGMS":"outputs/best_model_cahm_lgms.pt",
        "ALL":      "outputs/best_model_all.pt",
    })

CLI:
    python tools/evaluate_ablation.py --data_dir dataset --ckpts baseline:outputs/best_model.pt,CAHM:outputs/best_model_cahm.pt

คืนตาราง markdown พร้อม Val Acc / Balanced Acc / Needle_Holder↔Artery_Forceps error
"""
import argparse
import glob
import os
from typing import Dict, List, Optional

import numpy as np


def _find_pair_indices(class_names: List[str]):
    """หา index ของคู่ Needle_Holder ↔ Artery_Forceps แบบ tolerant (ชื่ออาจมี prefix)"""
    def find(key: str) -> Optional[int]:
        for i, n in enumerate(class_names):
            if key.lower() in n.lower():
                return i
        return None
    # ลองหลาย key เผื่อชื่อต่างกันเล็กน้อย
    needle = find("needle_holder") or find("needle")
    artery = find("artery_forceps") or find("artery")
    return needle, artery


def evaluate_single(ckpt_path: str, data_dir: Optional[str] = None) -> dict:
    """
    ประเมิน 1 checkpoint → dict
      accuracy, balanced_acc, cm, needle_error, report
    """
    from evaluate import evaluate_checkpoint
    # evaluate_checkpoint จะสร้าง confusion matrix + report ให้เสร็จ
    # show_plot=False เพื่อไม่เปิดหน้าต่างบน Colab
    res = evaluate_checkpoint(ckpt_path, data_dir=data_dir, show_plot=False, save_dir=None)
    # res มี: accuracy, balanced_accuracy, confusion_matrix, report, class_names
    # เติม needle↔artery error
    class_names: List[str] = res["class_names"] if "class_names" in res else res.get("classes", [])
    # fallback: ดูจาก res โดยตรง
    if not class_names:
        # evaluate_checkpoint คืน class_names ในบางเวอร์ชันเป็น "class_names"
        from evaluate import load_bundle
        bundle = load_bundle(ckpt_path)
        class_names = bundle["classes"]
    cm = res["confusion_matrix"]
    needle_idx, artery_idx = _find_pair_indices(class_names)
    needle_error = None
    needle_detail = ""
    if needle_idx is not None and artery_idx is not None:
        # นับทั้งสองทิศทาง
        a2n = int(cm[artery_idx, needle_idx]) if cm.shape[0] > max(needle_idx, artery_idx) else 0
        n2a = int(cm[needle_idx, artery_idx]) if cm.shape[0] > max(needle_idx, artery_idx) else 0
        needle_error = a2n + n2a
        needle_detail = f"{class_names[needle_idx]}↔{class_names[artery_idx]}: {n2a}+{a2n}={needle_error}"
    else:
        needle_detail = "คู่ Needle↔Artery ไม่พบ (ชื่อ class ไม่ตรง pattern)"

    return {
        "ckpt": ckpt_path,
        "class_names": class_names,
        "accuracy": float(res["accuracy"]),
        "balanced_acc": float(res.get("balanced_accuracy", res.get("balanced_acc", 0.0))),
        "cm": cm,
        "needle_error": needle_error,
        "needle_detail": needle_detail,
        "report": res.get("report", {}),
        "raw": res,
    }


def compare_checkpoints(ckpt_map: Dict[str, str], data_dir: Optional[str] = None,
                        save_csv: Optional[str] = None) -> List[dict]:
    """
    เปรียบเทียบหลาย checkpoint → พิมพ์ตาราง markdown + คืน list ผล

    ckpt_map: {"baseline": "outputs/best_model.pt", ...}
    """
    from sklearn.metrics import accuracy_score, balanced_accuracy_score  # noqa: F401 (used inside evaluate_single)
    results = []
    print("\n| Config | Val Acc | Balanced Acc | Needle↔Artery error | ckpt |")
    print("|---|---|---|---|---|")
    for name, path in ckpt_map.items():
        if not os.path.exists(path):
            print(f"| {name} | — | — | checkpoint ไม่พบ: {path} | {path} |")
            results.append({"name": name, "path": path, "found": False})
            continue
        try:
            r = evaluate_single(path, data_dir=data_dir)
            acc = r["accuracy"]
            bacc = r["balanced_acc"]
            needle = r["needle_detail"]
            print(f"| {name} | {acc:.4f} | {bacc:.4f} | {needle} | {path} |")
            results.append({"name": name, **r, "found": True})
        except Exception as e:
            print(f"| {name} | error | error | {e} | {path} |")
            results.append({"name": name, "path": path, "found": False, "error": str(e)})

    # สรุปว่า “คุ้มเก็บไว้” ตามเกณฑ์ plan Phase 3
    print("\n**เกณฑ์ตัดสิน (ตาม plan):** 1) ดีขึ้นจาก k-fold เฉลี่ย 2) ลด Needle↔Artery ได้จริง 3) ถ้าแย่ลงให้ตัดออก")
    if save_csv:
        try:
            import csv
            with open(save_csv, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["config", "val_acc", "balanced_acc", "needle_error", "ckpt"])
                for r in results:
                    if r.get("found"):
                        w.writerow([r["name"], f'{r["accuracy"]:.4f}', f'{r["balanced_acc"]:.4f}', r.get("needle_error", ""), r["ckpt"]])
                    else:
                        w.writerow([r["name"], "", "", "", r["path"]])
            print(f"[saved] {save_csv}")
        except Exception as e:
            print(f"[csv] skip: {e}")
    return results


def main(argv=None):
    ap = argparse.ArgumentParser(description="เปรียบเทียบ ablation checkpoints (Phase 3)")
    ap.add_argument("--data_dir", default="dataset", help="โฟลเดอร์ dataset (มี train/valid/_annotations.coco.json)")
    ap.add_argument("--ckpts", required=True,
                    help='เช่น "baseline:outputs/best_model.pt,CAHM:outputs/best_model_cahm.pt,LGMS:outputs/best_model_lgms.pt"')
    ap.add_argument("--pattern", default=None, help="หรือใช้ glob pattern เช่น 'outputs/best_model*.pt' (ชื่อ config จะเป็นชื่อไฟล์)")
    ap.add_argument("--save_csv", default=None, help="บันทึกผลเป็น CSV")
    args = ap.parse_args(argv)

    ckpt_map: Dict[str, str] = {}
    if args.pattern:
        for p in glob.glob(args.pattern):
            name = os.path.splitext(os.path.basename(p))[0]
            ckpt_map[name] = p
    if args.ckpts:
        for pair in args.ckpts.split(","):
            pair = pair.strip()
            if not pair:
                continue
            if ":" in pair:
                k, v = pair.split(":", 1)
                ckpt_map[k.strip()] = v.strip()
            else:
                ckpt_map[os.path.basename(pair)] = pair

    if not ckpt_map:
        ap.error("ไม่พบ checkpoint — ใส่ --ckpts หรือ --pattern")

    compare_checkpoints(ckpt_map, data_dir=args.data_dir, save_csv=args.save_csv)


if __name__ == "__main__":
    main()


## 3) Data sanity check

Check: images per class, mask overlay correctness, whether measured lengths look sensible — **always before training**

In [ ]:
CALIB_RATIO = None   # ← cm/pixel if you have a reference object (e.g. 0.05); None = use pixels

import sys; sys.path.insert(0, "/content")
from collections import Counter

from dataset import load_coco_records, visualize_records, compute_length_stats

train_recs, class_names = load_coco_records(DATA_DIR, "train")
dist = Counter(r["class_name"] for r in train_recs)
print(f"Total classes: {len(class_names)}")
for n in class_names:
    print(f"  {n:30s}: {dist[n]} images")

stats = compute_length_stats(train_recs, CALIB_RATIO)
unit = "cm" if CALIB_RATIO else "px"
print(f"\nlength stats (train): mean={stats[0]:.1f} {unit}, std={stats[1]:.1f}")

fig = visualize_records(train_recs, calibration_ratio=CALIB_RATIO, n=6, seed=7)

## 3.5) Check whether DINOv2 weights are ready (frozen kNN probe — no training needed)

Use frozen DINOv2 to extract features → predict with 1-nearest-neighbor:
- Accuracy far above random (1/14 ≈ 7%) (e.g. >40-50%) = features already separate classes well, worth training further
- Accuracy near random = domain gap too large → try `img_size=518` (sharper) or a larger backbone

In [ ]:
# ponytail: probe trains for zero steps — check feature quality before investing in full training
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from config import TrainConfig
from dataset import SurgicalInstrumentDataset, compute_length_stats
from model import SurgicalDinoFusion
from train import resolve_records

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
probe_model = SurgicalDinoFusion(finetune_mode="frozen").to(device).eval()

cfg0 = TrainConfig(data_dir=DATA_DIR)
tr_recs, va_recs, probe_classes = resolve_records(cfg0)
probe_stats = compute_length_stats(tr_recs, CALIB_RATIO)
flip_all = [True] * len(probe_classes)

@torch.no_grad()
def embed(recs, training):
    ds = SurgicalInstrumentDataset(recs, probe_stats, cfg0.img_size, CALIB_RATIO,
                                   flip_all if training else None, training,
                                   bbox_margin=cfg0.bbox_margin)
    E, Y = [], []
    for b in DataLoader(ds, batch_size=32, num_workers=2):
        out = probe_model.backbone(pixel_values=b["image"].to(device)).last_hidden_state
        E.append(out[:, 0].cpu())          # raw CLS — not through the fusion head (head is still randomly initialized)
        Y.append(b["label"])
    return torch.cat(E), torch.cat(Y)

Etr, ytr = embed(tr_recs, True)            # training-side augmentation enabled = free data augmentation for the probe
Eva, yva = embed(va_recs, False)
sim = F.normalize(Eva, dim=1) @ F.normalize(Etr, dim=1).T
pred = ytr[sim.argmax(dim=1)]
acc = (pred == yva).float().mean().item()
print(f"kNN probe accuracy: {acc:.3f}   (random = {1 / len(probe_classes):.3f})")
if acc < 0.30:
    print("⚠️ features barely separate classes — try img_size=518 or dinov2-base before full training")
else:
    print("✅ features look good — ready to train")

## 4) Train

- `finetune_mode="lora"` → train only LoRA adapters (very few parameters) to prevent overfitting
- Early stopping on validation loss; the best checkpoint is saved automatically
- For cross-validation evaluation → set `kfold=5`

In [ ]:
from config import TrainConfig
from train import run_training, run_kfold

cfg = TrainConfig(
    data_dir=DATA_DIR,
    img_size=504,            # 504=36×14 — from experiments (divisible by 14, best size in our tests)
    batch_size=32,
    finetune_mode="lora",     # "lora" (recommended) | "partial" | "frozen"
    epochs=50,
    num_workers=2,
    # ---- optional extras ----
    # calibration_ratio=CALIB_RATIO,
    # flip_allowed=["class_a", "class_b"],  # only classes that may be flipped (others are not flipped)
    # kfold=5,                              # Stratified 5-fold CV
)

if cfg.kfold:
    paths, accs = run_kfold(cfg)
    best_ckpt = paths[accs.index(max(accs))]   # pick the best fold
else:
    best_ckpt = run_training(cfg)

print("\nBest checkpoint:", best_ckpt)

## 5) Evaluate — accuracy + confusion matrix

Bright off-diagonal cells in the heatmap = class pairs the model confuses (often pairs that differ only in size)

In [ ]:
from evaluate import evaluate_checkpoint

metrics = evaluate_checkpoint(best_ckpt)
print(f"\nAccuracy: {metrics['accuracy']:.4f}")

## 5.5) Phase 3 — Train 6 ablation variants (baseline vs CAHM/LGMS/SEF)

Run each variant and keep its checkpoint — **do it this way on Colab** to compare results in the next cell.
Each variant early-stops at ~20-35 epochs (patience 12) | 504 px batch 32 on T4 ~7-8 GB — if OOM, batch is automatically reduced to 16.


In [ ]:
from config import TrainConfig
from train import run_training
import torch

ablation = {
    "baseline":  {},
    "cahm":      {"use_cahm": True},
    "lgms":      {"use_lgms": True},
    "sef":       {"use_sef": True},
    "cahm_lgms": {"use_cahm": True, "use_lgms": True},
    "all":       {"use_cahm": True, "use_lgms": True, "use_sef": True},
}

ckpt_map = {}
for name, extra in ablation.items():
    print(f"\n{'='*20} {name} {extra} {'='*20}")
    cfg_ab = TrainConfig(
        data_dir=DATA_DIR,
        img_size=504, batch_size=32,
        finetune_mode="lora", epochs=50, num_workers=2,
        output_dir=f"outputs_ablation/{name}",
        **extra
    )
    try:
        ckpt = run_training(cfg_ab)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("[OOM] Reducing batch 32 → 16 and retrying")
            torch.cuda.empty_cache()
            cfg_ab.batch_size = 16
            ckpt = run_training(cfg_ab)
        else:
            raise
    ckpt_map[name] = ckpt
    print(f"[{name}] ✅ {ckpt}")

print("\n--- ckpt_map ---")
for k,v in ckpt_map.items():
    print(f'{k}: "{v}"')

## 5.6) Compare ablations — Phase 3 table

Measure Val Acc / Balanced Acc / Needle_Holder↔Artery_Forceps error across all 6 variants
via `tools/evaluate_ablation.py` (runnable both in the notebook and as CLI)


In [ ]:
from tools.evaluate_ablation import compare_checkpoints

# Use ckpt_map from the previous cell; if you haven't run ablations, set paths manually:
# ckpt_map = {
#     "baseline": "outputs_ablation/baseline/best_model.pt",
#     "cahm": "outputs_ablation/cahm/best_model.pt",
#     ...
# }
results = compare_checkpoints(ckpt_map, data_dir=DATA_DIR, save_csv="outputs_ablation/ablation_results.csv")
print("\n--- Summary ---")
for r in results:
    if r.get("found"):
        print(f'{r["name"]:12s} acc={r["accuracy"]:.4f} bal={r["balanced_acc"]:.4f} {r["needle_detail"]}')

## 6) Inference demo — new image + mask → class + confidence

In [ ]:
from infer import load_pipeline, predict_record
from train import resolve_records

pack = load_pipeline(best_ckpt)
_, valid_recs, _ = resolve_records(cfg)   # try 3 validation images

for r in valid_recs[:3]:
    res = predict_record(pack, r)
    ok = "✅" if res["class"] == res["truth"] else "❌"
    tops = ", ".join(f'{t["class"]}:{t["prob"]:.2f}' for t in res["top3"])
    print(f'{ok} truth={res["truth"]:20s} pred={res["class"]:20s} conf={res["confidence"]:.3f}')
    print("      top3:", tops)

## 7) Save model back to Drive / Download

In [ ]:
# ▸ Download to local machine
from google.colab import files
files.download(best_ckpt)

# ▸ Or copy to Drive (mount first if not already mounted)
# !cp "{best_ckpt}" /content/drive/MyDrive/

---
### 💡 Tips
- **calibration_ratio**: place an object of known real length L (cm) under the same camera rig → `ratio = L / measure_length_px(mask)` — the length feature will be in cm and easier to interpret
- **handedness**: for left/right-handed classes → list only the classes that may be flipped in `flip_allowed`

### 🎯 Playbook for near-indistinguishable class pairs (e.g. Universal forceps 150 vs 151, Curved Root elevator vs Straight Root-tip elevator)
1. **Increase resolution first** — the head of forceps at 224 px is only ~20 px; set `img_size=518` (=37×14, accepted by DINOv2) and patch tokens will carry ~2.3× more "head" detail; if GPU memory is tight, reduce batch to 8
2. **Pairs that differ in length** (short curved elevator vs long straight) — the length feature already handles this, but the mask must cover the true tip (shadow included in mask = length error = feature error)
3. **Collect targeted extra data**: check the confusion matrix in step 5 → add more shots only for the confused pairs, rotating yaw by ~30° each time (some angles hide the head = you need angles that reveal the distinguishing point)
4. **Still heavily confused?** Plan B: two-stage — first separate coarse groups (forceps/elevator/probe…) then a pair-specific classifier on a crop around the head

### 📋 Checklist for the next annotation round (Roboflow COCO Segmentation)
- Polygon must cover **the whole instrument** including head and tip, but **exclude shadows** (shadows stretch minAreaRect)
- 1 annotation per instrument (multiple per image allowed — pipeline crops per instance via bbox_margin=0.15 from experiments)
- Do not rename classes after they are set (label = sorted name order)
- Keep ~25+ images per class and spread shooting angles evenly
- Green cloth: pipeline already has shadow augmentation, but shooting with even lighting is still best

- GPU: Runtime → Change runtime type → T4 GPU (CPU works but is slow); `img_size=518` + T4 + batch 8 ≈ fits
